# 03 — Results Visualisation

Load evaluation results from the DB and produce all standard figures.
Figures are also saved to `outputs/figures/<run_id>/` by `src/visualisation/export.py`.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

load_dotenv('../.env')
%matplotlib inline

from src.config import load_config
from src.db import db_connection, get_evaluations_for_run, list_runs
from src.visualisation.hallucination_rates import (
    plot_rates_by_type, plot_rates_by_model,
    plot_rates_by_strategy, plot_rates_by_dataset,
    plot_type_by_model,
)
from src.visualisation.heatmaps import (
    plot_model_strategy_heatmap, plot_all_datasets_heatmap, plot_type_heatmap
)
from src.visualisation.export import export_all_figures

cfg = load_config('../config/default.yaml')

In [ ]:
# Select the run to visualise
with db_connection(f'../{cfg.storage.db_path}') as conn:
    runs = list_runs(conn)
    
pd.DataFrame(runs)

In [ ]:
RUN_ID = runs[0]['run_id'] if runs else None
print('Visualising run:', RUN_ID)

with db_connection(f'../{cfg.storage.db_path}') as conn:
    evals = get_evaluations_for_run(conn, RUN_ID)

evals_df = pd.DataFrame(evals)
print(f'{len(evals_df)} evaluation rows loaded')
evals_df.head()

In [ ]:
# Overall hallucination rate
total = len(evals_df)
flagged = evals_df['any_hallucination'].sum()
print(f'Total narratives: {total}')
print(f'Flagged (any hallucination): {flagged} ({flagged/total*100:.1f}%)')
print()
for htype in ['sign_inversion','rank_swap','feature_fabrication','magnitude_distortion','omission']:
    if htype in evals_df.columns:
        n = evals_df[htype].sum()
        print(f'  {htype}: {n} ({n/total*100:.1f}%)')

## Bar charts

In [ ]:
fig = plot_rates_by_type(evals_df)
plt.show()

In [ ]:
fig = plot_rates_by_model(evals_df)
plt.show()

In [ ]:
fig = plot_rates_by_strategy(evals_df)
plt.show()

In [ ]:
fig = plot_type_by_model(evals_df)
plt.show()

## Heatmaps

In [ ]:
fig = plot_all_datasets_heatmap(evals_df)
plt.show()

In [ ]:
fig = plot_type_heatmap(evals_df)
plt.show()

## Export all figures to disk

In [ ]:
saved = export_all_figures(evals_df, cfg, RUN_ID)
print(f'\n{len(saved)} figures saved to outputs/figures/{RUN_ID}/')